In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings("ignore")


# ── STEP 1: GENERATE DATASET ──────────────────────────────────────────────────
def generate_dataset(n_samples: int = 1000) -> pd.DataFrame:
    """
    Generates a realistic synthetic crop disease dataset.
    In a real project this would be replaced with actual farm data.
    """

    print("📦 STEP 1 — Generating crop dataset...")

    np.random.seed(42)

    crops       = ["Wheat", "Rice", "Maize", "Cotton", "Soybean"]
    soil_types  = ["Sandy", "Loamy", "Clay", "Silty"]
    seasons     = ["Kharif", "Rabi", "Zaid"]

    data = {
        "crop"           : np.random.choice(crops, n_samples),
        "soil_type"      : np.random.choice(soil_types, n_samples),
        "season"         : np.random.choice(seasons, n_samples),
        "temperature_c"  : np.random.uniform(15, 45, n_samples).round(1),
        "humidity_pct"   : np.random.uniform(30, 95, n_samples).round(1),
        "rainfall_mm"    : np.random.uniform(0, 300, n_samples).round(1),
        "soil_ph"        : np.random.uniform(4.5, 8.5, n_samples).round(2),
        "nitrogen_level" : np.random.uniform(0, 100, n_samples).round(1),
        "crop_age_days"  : np.random.randint(10, 120, n_samples),
    }

    df = pd.DataFrame(data)

    # Disease risk logic — based on real agronomic conditions
    # High humidity + high temp + young crop = higher disease risk
    risk_score = (
        (df["humidity_pct"] > 75).astype(int) * 2 +
        (df["temperature_c"] > 30).astype(int) * 1 +
        (df["rainfall_mm"] > 150).astype(int) * 1 +
        (df["soil_ph"] < 5.5).astype(int) * 1 +
        (df["nitrogen_level"] > 80).astype(int) * 1 +
        (df["crop_age_days"] < 30).astype(int) * 1
    )

    # Add some noise to make it realistic
    noise = np.random.randint(0, 2, n_samples)
    df["disease_risk"] = ((risk_score + noise) >= 3).astype(int)
    # 0 = Low Risk, 1 = High Risk

    print(f"   ✅ Dataset created: {len(df)} records")
    print(f"   High Risk samples : {df['disease_risk'].sum()}")
    print(f"   Low Risk samples  : {(df['disease_risk'] == 0).sum()}")
    return df


# ── STEP 2: DATA CLEANING & PREPROCESSING ────────────────────────────────────
def preprocess_data(df: pd.DataFrame):
    """
    Cleans and prepares data for ML model.
    - Handles missing values
    - Encodes categorical columns
    - Splits into train/test sets
    """

    print("\n🔧 STEP 2 — Preprocessing data...")

    # Check for missing values
    missing = df.isnull().sum().sum()
    print(f"   Missing values found: {missing} (none in synthetic data)")

    # Encode categorical columns (crop, soil_type, season)
    le = LabelEncoder()
    categorical_cols = ["crop", "soil_type", "season"]

    df_encoded = df.copy()
    for col in categorical_cols:
        df_encoded[col] = le.fit_transform(df[col])

    print(f"   Categorical columns encoded: {categorical_cols}")

    # Features and target
    X = df_encoded.drop("disease_risk", axis=1)
    y = df_encoded["disease_risk"]

    # Train/test split — 80% train, 20% test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    print(f"   Training samples : {len(X_train)}")
    print(f"   Testing samples  : {len(X_test)}")
    print(f"   Features used    : {list(X.columns)}")

    return X_train, X_test, y_train, y_test, X.columns


# ── STEP 3: TRAIN ML MODEL ────────────────────────────────────────────────────
def train_model(X_train, y_train) -> RandomForestClassifier:
    """
    Trains a Random Forest classifier.
    Random Forest is ideal here — handles mixed data types,
    works well with small datasets, easy to interpret.
    """

    print("\n🤖 STEP 3 — Training Random Forest model...")

    model = RandomForestClassifier(
        n_estimators = 100,   # 100 decision trees
        max_depth    = 10,    # prevent overfitting
        random_state = 42
    )

    model.fit(X_train, y_train)
    print("   ✅ Model trained successfully.")
    return model


# ── STEP 4: EVALUATE MODEL ────────────────────────────────────────────────────
def evaluate_model(model, X_test, y_test, feature_names):
    """
    Tests model accuracy and prints detailed performance report.
    Also shows which features matter most for prediction.
    """

    print("\n📊 STEP 4 — Evaluating model performance...")

    y_pred   = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(f"\n   Model Accuracy: {accuracy * 100:.2f}%")
    print("\n   Detailed Report:")
    print("   " + "-"*45)

    report = classification_report(
        y_test, y_pred,
        target_names=["Low Risk", "High Risk"]
    )
    for line in report.split("\n"):
        print(f"   {line}")

    # Feature importance — what factors matter most
    importances  = model.feature_importances_
    feature_imp  = pd.Series(importances, index=feature_names)
    feature_imp  = feature_imp.sort_values(ascending=False)

    print("\n   🔍 Top Factors Affecting Disease Risk:")
    print("   " + "-"*45)
    for feat, imp in feature_imp.items():
        bar   = "█" * int(imp * 50)
        print(f"   {feat:<20} {bar} {imp:.3f}")

    return accuracy


# ── STEP 5: PREDICT NEW CROP ──────────────────────────────────────────────────
def predict_crop(model, crop_data: dict) -> str:
    """
    Takes real crop conditions as input and predicts disease risk.
    This is what a farmer or agronomist would use in practice.
    """

    # Encode categorical values manually
    crop_map      = {"Wheat": 4, "Rice": 3, "Maize": 1, "Cotton": 0, "Soybean": 2}
    soil_map      = {"Sandy": 2, "Loamy": 1, "Clay": 0, "Silty": 3}
    season_map    = {"Kharif": 0, "Rabi": 2, "Zaid": 1}

    input_data = pd.DataFrame([{
        "crop"           : crop_map.get(crop_data["crop"], 0),
        "soil_type"      : soil_map.get(crop_data["soil_type"], 1),
        "season"         : season_map.get(crop_data["season"], 0),
        "temperature_c"  : crop_data["temperature_c"],
        "humidity_pct"   : crop_data["humidity_pct"],
        "rainfall_mm"    : crop_data["rainfall_mm"],
        "soil_ph"        : crop_data["soil_ph"],
        "nitrogen_level" : crop_data["nitrogen_level"],
        "crop_age_days"  : crop_data["crop_age_days"],
    }])

    prediction   = model.predict(input_data)[0]
    probability  = model.predict_proba(input_data)[0]
    risk_label   = "🔴 HIGH RISK" if prediction == 1 else "🟢 LOW RISK"
    confidence   = probability[prediction] * 100

    print(f"\n   Prediction  : {risk_label}")
    print(f"   Confidence  : {confidence:.1f}%")
    print(f"   Low Risk    : {probability[0]*100:.1f}%")
    print(f"   High Risk   : {probability[1]*100:.1f}%")

    return risk_label


# ── MAIN PIPELINE ─────────────────────────────────────────────────────────────
def run_pipeline():
    print("="*60)
    print("  CROP DISEASE RISK PREDICTOR")
    print("  Kisan Udyog — AI/ML Internship Project")
    print("  Built by: Aadish Garg")
    print("="*60)

    # Step 1 — Data
    df = generate_dataset(n_samples=1000)

    # Step 2 — Preprocess
    X_train, X_test, y_train, y_test, feature_names = preprocess_data(df)

    # Step 3 — Train
    model = train_model(X_train, y_train)

    # Step 4 — Evaluate
    accuracy = evaluate_model(model, X_test, y_test, feature_names)

    # Step 5 — Predict on sample crops
    print("\n🌾 STEP 5 — Testing predictions on sample crops...")
    print("="*60)

    # Sample 1 — High risk conditions
    print("\n   Sample 1: Rice in monsoon with high humidity")
    predict_crop(model, {
        "crop"          : "Rice",
        "soil_type"     : "Clay",
        "season"        : "Kharif",
        "temperature_c" : 35,
        "humidity_pct"  : 88,
        "rainfall_mm"   : 210,
        "soil_ph"       : 5.2,
        "nitrogen_level": 85,
        "crop_age_days" : 20,
    })

    # Sample 2 — Low risk conditions
    print("\n   Sample 2: Wheat in dry season with normal conditions")
    predict_crop(model, {
        "crop"          : "Wheat",
        "soil_type"     : "Loamy",
        "season"        : "Rabi",
        "temperature_c" : 22,
        "humidity_pct"  : 45,
        "rainfall_mm"   : 50,
        "soil_ph"       : 6.8,
        "nitrogen_level": 40,
        "crop_age_days" : 60,
    })

    # Save dataset for reference
    df.to_csv("crop_disease_dataset.csv", index=False)
    print("\n💾 Dataset saved → crop_disease_dataset.csv")

    print("\n" + "="*60)
    print(f"  ✅ PIPELINE COMPLETE — Model Accuracy: {accuracy*100:.2f}%")
    print("="*60)


# ── RUN ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    run_pipeline()
